# EZ Car Rental: Dynamic Pricing with Thompson Sampling
### ISBA 2415 - Reinforcement Learning - M5 Project

This notebook implements **Thompson Sampling** and **Epsilon-Greedy** agents for the EZ Car Rental dynamic pricing problem. We compare both against an oracle with perfect knowledge of demand.

**State space:** (city, time_category, utilization) → 20 states  
**Action space:** {$3, $5, $7, $9, $11}/hr → 5 prices  
**Total state-action pairs:** 100

In [ ]:
import csv
import random
import os
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime

random.seed(42)

# Load journeys data
JOURNEYS_PATH = os.path.join("data", "journeys.csv")

def load_journeys(path):
    rows = []
    with open(path, "r") as f:
        reader = csv.DictReader(f)
        for row in reader:
            price = float(row["Trip Sum Trip Price"].replace("$", "").replace(",", ""))
            start = datetime.strptime(row["Trip Start At Local Time"], "%Y-%m-%d %H:%M:%S")
            end = datetime.strptime(row["Trip End At Local Time"], "%Y-%m-%d %H:%M:%S")
            duration_hrs = max((end - start).total_seconds() / 3600, 0.5)
            rows.append({
                "city": row["Car Parking Address City"].strip(),
                "hour": start.hour,
                "hourly_rate": price / duration_hrs,
            })
    return rows

journeys = load_journeys(JOURNEYS_PATH)
print(f"Loaded {len(journeys)} trips")

# Top 5 cities
city_counts = defaultdict(int)
for j in journeys:
    city_counts[j["city"]] += 1
CITIES = sorted(city_counts, key=city_counts.get, reverse=True)[:5]
journeys = [j for j in journeys if j["city"] in CITIES]
print(f"Top 5 cities: {CITIES}")
print(f"Filtered to {len(journeys)} trips")

## State & Action Space Definition

In [ ]:
PRICES = [3, 5, 7, 9, 11]
city_to_idx = {c: i for i, c in enumerate(CITIES)}
NUM_STATES = len(CITIES) * 2 * 2  # 5 cities x 2 time x 2 util = 20

def idx_to_state(idx):
    return (idx // 4, (idx % 4) // 2, idx % 2)

def state_label(s):
    return f"({CITIES[s[0]]}, {'Peak' if s[1] else 'Off-peak'}, {'High-util' if s[2] else 'Low-util'})"

print(f"States: {NUM_STATES}, Actions: {len(PRICES)}, Pairs: {NUM_STATES * len(PRICES)}")

## Build Simulator from Data

We compute empirical booking probabilities from the Journeys CSV. For each state, the booking probability at price $p$ is the fraction of observed hourly rates ≥ $p$. The agent never sees this data directly.

In [ ]:
def compute_empirical_booking_rates(journeys):
    groups = defaultdict(list)
    for j in journeys:
        ci = city_to_idx[j["city"]]
        tc = 1 if 7 <= j["hour"] <= 21 else 0
        groups[(ci, tc)].append(j["hourly_rate"])

    booking_probs = {}
    for (ci, tc), rates in groups.items():
        median_rate = sorted(rates)[len(rates) // 2]
        low = [r for r in rates if r < median_rate]
        high = [r for r in rates if r >= median_rate]
        for ul, subset in [(0, low), (1, high)]:
            if not subset:
                subset = rates
            for p in PRICES:
                prob = max(0.05, min(0.95, sum(1 for r in subset if r >= p) / len(subset)))
                booking_probs[((ci, tc, ul), p)] = prob
    return booking_probs

TRUE_PROBS = compute_empirical_booking_rates(journeys)

def simulate_booking(state, price, rng=None):
    return 1 if (rng or random).random() < TRUE_PROBS.get((state, price), 0.3) else 0

# Show true probabilities for first 3 cities
print(f"{'State':<45} {'$3':>6} {'$5':>6} {'$7':>6} {'$9':>6} {'$11':>6}")
print("-" * 75)
for si in range(12):
    s = idx_to_state(si)
    probs = [TRUE_PROBS.get((s, p), 0) for p in PRICES]
    print(f"{state_label(s):<45} {probs[0]:>6.3f} {probs[1]:>6.3f} {probs[2]:>6.3f} {probs[3]:>6.3f} {probs[4]:>6.3f}")

## Agent Implementations

### Thompson Sampling
- Maintains `Beta(α, β)` posterior for each state-action pair
- Samples θ from posterior, picks action maximizing `price × θ`
- Update: `α += reward`, `β += (1 - reward)`

### Epsilon-Greedy
- With probability ε=0.15, pick random price
- Otherwise, pick price with highest estimated revenue

In [ ]:
class ThompsonSamplingAgent:
    def __init__(self):
        self.alpha = {}
        self.beta_param = {}
        for si in range(NUM_STATES):
            s = idx_to_state(si)
            for a in PRICES:
                self.alpha[(s, a)] = 1.0
                self.beta_param[(s, a)] = 1.0

    def select_action(self, state, rng=None):
        r = rng or random
        best_action, best_value = None, -1
        for a in PRICES:
            theta = r.betavariate(self.alpha[(state, a)], self.beta_param[(state, a)])
            ev = a * theta
            if ev > best_value:
                best_value, best_action = ev, a
        return best_action

    def update(self, state, action, reward):
        self.alpha[(state, action)] += reward
        self.beta_param[(state, action)] += (1 - reward)

    def get_learned_probs(self):
        return {k: self.alpha[k] / (self.alpha[k] + self.beta_param[k]) for k in self.alpha}

    def get_policy(self):
        policy = {}
        for si in range(NUM_STATES):
            s = idx_to_state(si)
            best_price, best_rev = None, -1
            for a in PRICES:
                ev = a * self.alpha[(s, a)] / (self.alpha[(s, a)] + self.beta_param[(s, a)])
                if ev > best_rev:
                    best_rev, best_price = ev, a
            policy[s] = best_price
        return policy


class EpsilonGreedyAgent:
    def __init__(self, epsilon=0.15):
        self.epsilon = epsilon
        self.counts = {}
        self.sum_rewards = {}
        for si in range(NUM_STATES):
            s = idx_to_state(si)
            for a in PRICES:
                self.counts[(s, a)] = 0
                self.sum_rewards[(s, a)] = 0.0

    def select_action(self, state, rng=None):
        r = rng or random
        if r.random() < self.epsilon:
            return r.choice(PRICES)
        best_action, best_rev = None, -1
        for a in PRICES:
            n = self.counts[(state, a)]
            if n == 0:
                return a
            ev = a * self.sum_rewards[(state, a)] / n
            if ev > best_rev:
                best_rev, best_action = ev, a
        return best_action

    def update(self, state, action, reward):
        self.counts[(state, action)] += 1
        self.sum_rewards[(state, action)] += reward

    def get_policy(self):
        policy = {}
        for si in range(NUM_STATES):
            s = idx_to_state(si)
            best_price, best_rev = None, -1
            for a in PRICES:
                n = self.counts[(s, a)]
                if n == 0: continue
                ev = a * self.sum_rewards[(s, a)] / n
                if ev > best_rev:
                    best_rev, best_price = ev, a
            policy[s] = best_price if best_price else PRICES[0]
        return policy

print("Agents defined.")

## Oracle Policy & Simulation

In [ ]:
# Oracle: optimal policy with full knowledge
ORACLE_POLICY, ORACLE_REV = {}, {}
for si in range(NUM_STATES):
    s = idx_to_state(si)
    best_price, best_rev = None, -1
    for p in PRICES:
        rev = p * TRUE_PROBS.get((s, p), 0)
        if rev > best_rev:
            best_rev, best_price = rev, p
    ORACLE_POLICY[s] = best_price
    ORACLE_REV[s] = best_rev

# Run an agent for N episodes
NUM_EPISODES = 5000
EVAL_INTERVAL = 100

def run_agent(agent, seed=42):
    rng = random.Random(seed)
    ep_rev, ep_oracle, regret, policy_acc = [], [], [], []
    run_rev, run_oracle = 0, 0

    for ep in range(NUM_EPISODES):
        state = idx_to_state(rng.randint(0, NUM_STATES - 1))
        price = agent.select_action(state, rng=rng)
        booked = simulate_booking(state, price, rng=rng)
        rev = price * booked
        agent.update(state, price, booked)

        run_rev += rev
        run_oracle += ORACLE_REV[state]
        ep_rev.append(rev)
        ep_oracle.append(ORACLE_REV[state])
        regret.append(run_oracle - run_rev)

        if (ep + 1) % EVAL_INTERVAL == 0:
            pol = agent.get_policy()
            acc = sum(1 for si in range(NUM_STATES) if pol[idx_to_state(si)] == ORACLE_POLICY[idx_to_state(si)]) / NUM_STATES
            policy_acc.append((ep + 1, acc))

    return {"ep_rev": ep_rev, "ep_oracle": ep_oracle, "regret": regret, "policy_acc": policy_acc,
            "total_rev": sum(ep_rev), "total_oracle": run_oracle}

# Run both agents
ts_agent = ThompsonSamplingAgent()
ts = run_agent(ts_agent, seed=42)
print(f"Thompson Sampling: ${ts['total_rev']:,.0f} / ${ts['total_oracle']:,.0f} ({ts['total_rev']/ts['total_oracle']:.1%})")

eg_agent = EpsilonGreedyAgent(epsilon=0.15)
eg = run_agent(eg_agent, seed=42)
print(f"Epsilon-Greedy:    ${eg['total_rev']:,.0f} / ${eg['total_oracle']:,.0f} ({eg['total_rev']/eg['total_oracle']:.1%})")

## Results: Cumulative Regret

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(range(NUM_EPISODES), ts["regret"], color='#2980b9', linewidth=1.5, label='Thompson Sampling')
ax.plot(range(NUM_EPISODES), eg["regret"], color='#e67e22', linewidth=1.5, label='Epsilon-Greedy (0.15)')
ax.set_xlabel("Episode"); ax.set_ylabel("Cumulative Regret ($)")
ax.set_title("Cumulative Regret: Thompson Sampling vs Epsilon-Greedy")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Results: Policy Accuracy Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ts_ep, ts_acc = zip(*ts["policy_acc"])
eg_ep, eg_acc = zip(*eg["policy_acc"])
ax.plot(ts_ep, [a*100 for a in ts_acc], color='#2980b9', linewidth=2, marker='o', markersize=2, label='Thompson Sampling')
ax.plot(eg_ep, [a*100 for a in eg_acc], color='#e67e22', linewidth=2, marker='s', markersize=2, label='Epsilon-Greedy')
ax.axhline(y=100, color='green', linestyle='--', alpha=0.4, label='Perfect')
ax.set_xlabel("Episode"); ax.set_ylabel("Policy Accuracy (%)")
ax.set_title("Policy Accuracy Over Training")
ax.set_ylim(0, 105); ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Results: Revenue Comparison

In [ ]:
window = 200
def windowed_avg(data, w):
    return [sum(data[i:i+w])/w for i in range(0, len(data)-w+1, w)]

ts_wavg = windowed_avg(ts["ep_rev"], window)
eg_wavg = windowed_avg(eg["ep_rev"], window)
or_wavg = windowed_avg(ts["ep_oracle"], window)
xs = [i*window+1 for i in range(len(ts_wavg))]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(xs, or_wavg, color='green', linewidth=2, linestyle='--', label='Oracle')
ax.plot(xs, ts_wavg, color='#2980b9', linewidth=2, label='Thompson Sampling')
ax.plot(xs, eg_wavg, color='#e67e22', linewidth=2, label='Epsilon-Greedy')
ax.set_xlabel("Episode Window Start"); ax.set_ylabel("Avg Revenue per Episode ($)")
ax.set_title("Revenue Comparison Across Methods")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Final Policy Table

In [ ]:
ts_pol = ts_agent.get_policy()
eg_pol = eg_agent.get_policy()

ts_match = sum(1 for si in range(NUM_STATES) if ts_pol[idx_to_state(si)] == ORACLE_POLICY[idx_to_state(si)])
eg_match = sum(1 for si in range(NUM_STATES) if eg_pol[idx_to_state(si)] == ORACLE_POLICY[idx_to_state(si)])

print(f"{'State':<42} {'TS':>4} {'EG':>4} {'Oracle':>7}")
print("-" * 60)
for si in range(NUM_STATES):
    s = idx_to_state(si)
    ts_m = "*" if ts_pol[s] != ORACLE_POLICY[s] else " "
    eg_m = "*" if eg_pol[s] != ORACLE_POLICY[s] else " "
    print(f"{state_label(s):<42} ${ts_pol[s]:>2}{ts_m} ${eg_pol[s]:>2}{eg_m}  ${ORACLE_POLICY[s]:>2}")

print(f"\nThompson Sampling accuracy: {ts_match}/{NUM_STATES} ({ts_match/NUM_STATES:.0%})")
print(f"Epsilon-Greedy accuracy:   {eg_match}/{NUM_STATES} ({eg_match/NUM_STATES:.0%})")
print(f"\n{'Metric':<30} {'Thompson':>12} {'Eps-Greedy':>12}")
print("-" * 55)
print(f"{'Total Revenue':<30} ${ts['total_rev']:>10,.0f} ${eg['total_rev']:>10,.0f}")
print(f"{'Revenue Ratio':<30} {ts['total_rev']/ts['total_oracle']:>11.1%} {eg['total_rev']/eg['total_oracle']:>11.1%}")
print(f"{'Total Regret':<30} ${ts['regret'][-1]:>10,.0f} ${eg['regret'][-1]:>10,.0f}")